# Core 02 - Native Skills

Objetivo: empaquetar Tools, prompts, contrato y policy como una capacidad reusable mediante `toolkit.skill`.

**Lugar en el modelo:** una `Skill` empaqueta Tools, instrucciones, contratos, policy y assets como capacidad reusable.

**Evidencia exigida:** la Skill debe validarse, ejecutarse desde un Agent y poder cargarse desde el contrato de filesystem de Agentic Systems.

**Límite de la evidencia:** `toolkit.skill` no afirma compatibilidad automática con cualquier formato de Anthropic/Claude o ChatGPT/OpenAI; un paquete externo es compatible sólo si se adapta al contrato `Skill` o al loader `SKILL.md + skill.py`.

## Parametros de la demostracion

| Parametro | Default | Proposito |
|---|---|---|
| AGENTIC_SYSTEMS_DEMO_SYMBOL | skill | Entrada del usuario para las Tools de la Skill. |
| tools | API inspectors | Capacidades reales agrupadas por toolkit.skill. |
| runtime | python-runtime | Ejecucion reproducible antes de cambiar provider. |

In [ ]:
import os

import agentic_systems as toolkit

SYMBOL = os.getenv("AGENTIC_SYSTEMS_DEMO_SYMBOL", "skill")

## 1) Tools de la capacidad

In [ ]:
@toolkit.tool
def inspect_public_api(symbol: str) -> dict:
    return {"symbol": symbol, "is_public": symbol in toolkit.__all__}

@toolkit.tool
def package_version() -> dict:
    return {"package_version": toolkit.__version__}

## 2) Crear y validar la Skill

La Skill conserva componentes declarativos; no ejecuta un provider al construirse.

In [ ]:
inspection_skill = toolkit.skill(
    name="public_api_inspection",
    version="1.0.0",
    description="Inspecciona la superficie publica instalada.",
    tools=[inspect_public_api, package_version],
    prompts={"instructions": "Usa Tools para responder con evidencia instalada."},
    contracts={"default": toolkit.AgentContract(must_call=["inspect_public_api"]).model_dump(mode="json")},
    policy=toolkit.RunPolicy(max_tool_calls=1, max_turns=2).model_dump(mode="json"),
    metadata={"domain": "public-api"},
)
validation = inspection_skill.check()
assert validation.ok
toolkit.show_json({
    "info": inspection_skill.info(),
    "validation": validation.to_dict(),
    "tool_names": inspection_skill.tool_names,
}, title="Skill contract")

## 3) Consumir la Skill desde un Agent

In [ ]:
runtime = toolkit.runtime(provider="python-runtime")
agent = toolkit.agent(
    name="skill_consumer",
    instructions=inspection_skill.instructions,
    skills=[inspection_skill],
    runtime=runtime,
    contract=toolkit.AgentContract(must_call=["inspect_public_api"]),
)
result = agent.run(
    {"tool": "inspect_public_api", "input": {"symbol": SYMBOL}},
    mode="eval",
)
assert result.ok, result.errors
assert result.engine == "python-runtime"
toolkit.human_result(result, title="Skill Agent RunResult", show_lineage=True)

## 4) Cargar una Skill nativa desde carpeta

El formato nativo ejecutable conserva SKILL.md, skill.py y build_skill() -> Skill.
La carga es explicita: no hay ZIP, YAML externo, scripts genericos ni activacion automatica.


Este loader implementa el contrato nativo de Agentic Systems. Un `SKILL.md` de Claude/Anthropic o una Skill de ChatGPT/OpenAI puede aportar una intención parecida, pero no se considera intercambiable por nombre: debe incluir el adaptador ejecutable esperado o convertirse explícitamente a `Skill`.

In [ ]:
folder_skill = toolkit.load_skill("tutorials/skills/tutorial_api_inspection")
assert isinstance(folder_skill, toolkit.Skill)
assert folder_skill.check().ok
toolkit.show_json(
    {
        "name": folder_skill.name,
        "tool_names": folder_skill.tool_names,
        "instructions": folder_skill.instructions,
    },
    title="Native folder Skill",
)


## 5) API realmente ejercitada

In [ ]:
api_coverage = [
    "toolkit.tool", "toolkit.skill", "toolkit.load_skill", "toolkit.Skill", "Skill.check", "Skill.info", "Skill.tool_names",
    "toolkit.runtime", "toolkit.agent", "agent.run", "toolkit.human_result",
    "toolkit.AgentContract",
    "toolkit.RunPolicy",
    "toolkit.show_json",
]
toolkit.show_json(api_coverage, title="Skill API coverage")

## Resultado e interpretacion

Una Skill valida y un Agent que ejecuta una Tool registrada por la Skill.